# Auswertung ABS-Bremsung

Bestimmung des Bremsweges und der mittleren Vollverzögerung einer ABS-Bremsung aus der gemessenen Fahrzeuggeschwindigkeit.

Dies ist ein Beispiel für Messdatenauswertung mit _numpy_ und Visualisierung mit _matplotlib_

<hr style="border:2px solid gray"> </hr>

2025-05-06 ug Version 1.6

<hr style="border:2px solid gray"> </hr>

Verfeinerung: Bestimmung der Werte durch Interpolation zwischen den Abtastpunkten

## Schritt 1: Vorbereitung der Python Umgebung

Einstellung für Matplotlib zur Ausgabe der Plots:

    %matplotlib inline   -> just show the plots
    %matplotlib notebook -> create interactive plots   - funktioniert nicht in Jupyterlab

In [ ]:
#%matplotlib inline
#%matplotlib notebook

Importieren der Python Pakete _numpy_ und _matplotlib_:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

<hr style="border:2px solid gray"> </hr>

## Schritt 2: Messung einlesen

Die Messung liegt als Textdatei im csv-Format (Comma Separate Value) vor.
Die Daten sind tabellenförmig als Text abgespeichtert. 
Die Werte der Spalten sind durch ein Komma als Trennzeichen voneinander angegrenzt.
In der vorliegenden Datei wurde ein Semikolon `;` als Trennzeichen verwendet.
Die erste Zeile enthält statt Zahlen die Namen der Spalten.

Bedeutung der Spalten:
- Time: Zeitachse [s]
- BrkPedPos: Bremspedalposition in Prozenz: 0..100 [%]
- BrakeSwitch: Bremspedalschalter 0: Bremspedal nicht betätigt, 1: Bremspedal betätigt [bool]
- ABSActive:  Eingriff des Anti-lock Braking Systems; 0: kein ABS Eingriff, 1: ABS Eingriff [bool]
- ABSFullyOp: ABS Betriebsbereitsschaft; 0: nicht betriebsbereit, 1: betriebsbereit [bool]
- ASRBrkCntrlActive: Antriebs-Schlupf Regelung - Bremseingriff - 0: nicht aktiv, 1: aktiv [bool]
- ASREngCntrlActive: Antriebs-Schlupf Regelung - Motoreingriff   0: nicht aktiv, 1: aktiv [bool]
- FrontAxleSpeed: Fahrzeuggeschwindigkeit: mittlere Geschwindigkeit der Räder der Vorderachse [km/h]
- v_FL: Geschwindigkeit linkes Vorderrad [km/h]
- v_FR: Geschwindigkeit rechtes Vorderrad [km/h]
- v_RL1: Geschwindigkeit linkes Rad der ersten Hinterachse [km/h]
- v_RR1: Geschwindigkeit rechtes Rad der ersten Hinterachse [km/h]
- v_RL2: Geschwindigkeit linkes Rad der zweiten Hinterachse [km/h]
- v_RR2: Geschwindigkeit rechtes Rad der zweiten Hinterachse [km/h]

Die csv-Datei kann in einem Text-Editor wie Visual Studio Code oder Notepad++ als Text angezeigt werden.
<br>
In Visual Studio Code es Extensions z.B. Rainbow CSV, um die Darstellung von csv-Dateien zu verbessern.
<br>
Die csv-Datei kann aber auch in Microsoft Excel geladen werden.

Die Datei liegt im Verzeichnis wie dieses Jupyter Notebokk
<br>
Dateiname der csv-Datei:

In [ ]:
csvFilename = r'ABS_Bremsung.csv'

###  Einlesen der csv-Datei in ein Numpy Array

Mit der Numpy-Funktion [np.genfromtxt()](https://numpy.org/doc/stable/reference/generated/numpy.genfromtxt.html)
wird der Inhalt der csv-Datei in ein Numpy Array eingelesen. 
<br>
Mit dem Argument `dtype=float` wird der Datentype des zu erzeugenden Numpy Array als Datentyp float festgelegt.
<br>
Mit dem Argument `delimiter=';'` wird das Semikolon als Trennzeichen zwischen den Werte festgelegt.
<br>
Mit dem Argumen `names=True` wird die erste Zeile der csv-Datei als Spaltename eingelesen.
<br>
Referenz:
[Read CSV file to numpy array, first row as strings, rest as float - stackoverflow](https://stackoverflow.com/questions/12336234/read-csv-file-to-numpy-array-first-row-as-strings-rest-as-float)

In [ ]:
data = np.genfromtxt(csvFilename, dtype=float, delimiter=';', names=True) 

In [ ]:
data

`np.genfromtxt()` erzeugt ein Numpy Array 

In [ ]:
type(data)

Dieses Numpy Array hat die Dimension 1, obwohl die csv-Datei eine Tabelle darstellt:

In [ ]:
data.ndim

Diese eine Dimension enthält 193 Elemente:

In [ ]:
data.shape

Der Datentyp des Numpy Array ist ein [_Strukturiertes Array_](https://numpy.org/doc/stable/user/basics.rec.html) und enthält die Werte der einzelnen Spalten.

Das `dtype`-Attribut eines Structured Arrays gibt dir die komplette Struktur des Datentyps zurück, inklusive aller Feldnamen und deren Typen:

<!--
Perplexity - 2025-05-31 Wie kann ich den Datentype eines numpy struktured array weiter analysieren?
https://www.perplexity.ai/search/wie-kann-ich-den-datentype-ein-ytjyodZhRiCjAH7Kctpu4g#0
-->

In [ ]:
data.dtype

So siehst du, welche Felder existieren und welchen Typ sie haben.
<br>
<br>
Feldnamen und Felddatentypen auslesen
<br>
Du kannst die Feldnamen und die zugehörigen Typen direkt über das `dtype`-Objekt abfragen:
<br>
Zugriff auf die Feldnamen:


In [ ]:
data.dtype.names

Typ eines bestimmten Feldes ausgeben

In [ ]:
data.dtype['Time']

Iteration über die Namen im Feld und Ausgabe des zu gehörigen Datentyps:

In [ ]:
for name in data.dtype.names:
    print(f"{name}: {data.dtype[name]}")

Der Zugriff auf die Elemente des strukturierten Arrays ähnelt dem eines Python Dictionaries; 

Beispiel Zeitachse `'Time'`:

In [ ]:
data['Time']

In [ ]:
type(data['Time'])

Beispiel: Fahrzeuggeschwindigkeit `'FrontAxleSpeed'`

In [ ]:
data['FrontAxleSpeed']

In [ ]:
type(data['FrontAxleSpeed'])

Damit haben wir alles, um auf die Messwerte zugreifen zu können.

<hr style="border:2px solid gray"> </hr>

## Schritt 3: Messsignale sichten

#### Zeitachse - `data['Time']`

In [ ]:
t = data['Time']
t.shape

Die Zeitachse enthält 193 Abtastpunkte.
<br>
Der Start und das Ende der Zeitachse sind:

In [ ]:
t.min(), t.max()

#### Bestimmung der Abtastzeit `dt`

Ansehen der ersten 10 Abtastpunkte:

In [ ]:
t[:10]

Es sieht danach aus, dass eine konstante Abtastzeit von 50ms vorliegt.

Jetzt wollen wir die Abtastzeit aus den vorliegen Daten bestimmen.

Zur Berechnung von diskreten Differenzen zwischen den Abtastzeitpunkten kann die Numpy Funktion [np.diff()](https://numpy.org/doc/stable/reference/generated/numpy.diff.html) verwendet werden.

Im folgenden berechnen wir die diskreten Differenzen der gesamten Zeitachse, wählen aber nur die ersten 10 aus.

In [ ]:
np.diff(t)[:10]

Als nächstes berechnen wir den Mittelwert der diskreten Differenzen zwischen den Abtastzeitpunkten mit der Numpy Funktion [np.mean()](https://numpy.org/doc/stable/reference/generated/numpy.mean.html) 

In [ ]:
np.mean(np.diff(t))

Das entspricht unseren ersten Vermutungen.

Zur Überprüfung berechnen wir die Standardabweichung der diskreten Differenzen zwischen den Abtastzeitpunkten mit der Numpy Funktion [np.std()](https://numpy.org/doc/stable/reference/generated/numpy.std.html)

In [ ]:
np.std(np.diff(t))

Die Standardabweichung ist sehr klein, die Abtastzeiten sind konstant und so stimmt die aus dem Mittelwert berechnete mittlere Abtastzeit sehr gut.

Bestimmung der mittleren Abtastzeit in einer Zeile und Zuweisen zur Variablen `dt`:

In [ ]:
dt = np.mean(np.diff(t))
print(f"Abtastzeit dt = {dt:.3f}s oder {dt*1000:.1f} ms")

#### Bremspedal `data['BrkPedPos']`

In [ ]:
BrkPedPos = data['BrkPedPos']
BrkPedPos.shape

#### Bremspedal-Schalter data['BrakeSwitch']

In [ ]:
BrakeSwitch = data['BrakeSwitch']
BrakeSwitch.shape

#### Fahrzeug-Geschwindigkeit `data['FrontAxleSpeed']` in km/h 

In [ ]:
v = data['FrontAxleSpeed']
v.shape

<hr style="border:2px solid gray"> </hr>

## Schritt 4: Plotten der Signale  mit Matplotlib

Referenz
[figure title](https://matplotlib.org/stable/gallery/subplots_axes_and_figures/figure_title.html)


`constrained_layout=True` sorgt dafür, dass Matplotlib die Abstände zwischen Figuren, Achsen, Titeln und Legenden automatisch so anpasst, dass sich Elemente nicht überlappen. Es verbessert den Abstand von Subplots und sorgt für sauberere Layouts ohne manuellen `tight_layout()` Aufruf.

In [ ]:
fig,(ax1,ax2) = plt.subplots(ncols=1,nrows=2, figsize=(16,8), sharex=True,constrained_layout=True)

fig.suptitle('ABS-Bremsung', fontsize=16)

# erster Subplot
ax1.set_title('Bremsschalter und Bremspedal')
ax1.plot(t,BrakeSwitch*100,label='Bremsschalter *100')
ax1.plot(t,BrkPedPos,label='Bremspedal 0..100%')
ax1.legend()
ax1.grid(True)
ax1.set_ylabel('[%]')


# zweiter Subplot
ax2.set_title('Fahrzeuggeschwindigkeit')
ax2.plot(t,v,label='Fahrzeuggeschwindigkeit')
ax2.legend()
ax2.grid(True)
ax2.set_ylabel('[km/h]')

ax2.set_xlabel('Zeit [s]')

# Abspeichern als  png-Bild
plt.savefig('ABS-Bremsung-Überblick.png', facecolor='white')


<hr style="border:2px solid gray"> </hr>

## Schritt 5: Berechnung des zurückgelegten Weges 

Der zurückgelegte Weg wird vereinfacht durch Aufsummieren der zurückgelegten Weginkremente aus jedem Abtastschritt: Momentan-Geschwindigkeit * Abtastzeit bestimmt.

Eine kummulierte Summe kann mit der Numpy Funktion [np.cumsum](https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html) berechnet werden.

Zunächst die ersten Werte der Geschwindigkeit darstellen:

In [ ]:
v[:10]

Berechnung des zurückgelegten Weges:
<br>
Die Geschwindigkeit muss von km/h in m/s umrechnet werden; deshalb der Faktor 3.6.

Umrechnen km/h in m/s
<br>
$
\frac{1 km}{1 h} = \frac{1000 m}{3600 s} = \frac{1}{3.6} \frac{1 m}{1 s}
$

In [ ]:
(v/3.6*dt)[:10]

Weg vom Start der Messung

In [ ]:
s_vom_start_der_Messung = np.cumsum(v)/3.6*dt
s_vom_start_der_Messung[:10]

In [ ]:
fig,(ax1,ax2,ax3) = plt.subplots(ncols=1,nrows=3, figsize=(16,8), sharex=True,constrained_layout=True)

fig.suptitle('ABS-Bremsung - {}', fontsize=16)

# erster Subplot
ax1.set_title('Bremsschalter und Bremspedal')
ax1.plot(t,BrakeSwitch*100,label='Bremsschalter *100')
ax1.plot(t,BrkPedPos,label='Bremspedal 0..100%')
ax1.legend()
ax1.grid(True)
ax1.set_ylabel('[%]')


# zweiter Subplot
ax2.set_title('Fahrzeuggeschwindigkeit')
ax2.plot(t,v,label='Fahrzeuggeschwindigkeit')
ax2.legend()
ax2.grid(True)
ax2.set_ylabel('[km/h]')

# dritter Subplot
ax3.set_title('zurückgelegter Weg')
ax3.plot(t,s_vom_start_der_Messung,label='Weg')
ax3.legend()
ax3.grid(True)
ax3.set_ylabel('[m]')

ax3.set_xlabel('Zeit [s]')

# Abspeichern als  png-Bild
plt.savefig('ABS-Bremsung-Überblick mit Weg.png', facecolor='white')


## Schritt 6: Ermittung des Bremsweges und der mittleren Vollverzögerung

ECE R13 [Link](https://eur-lex.europa.eu/legal-content/DE/TXT/PDF/?uri=CELEX:42016X0218(01)&from=en)

Vorschrift zur Ermittlung des Bremsweges und der mittleren Vollverzögerung:

<img  src=".\ECE_R13_Bremsweg.png" width=600>

### Bestimmung des Startzeitpunkte - wenn der Fahrer auf das Bremspedal getreten ist


**Einschub:** Ermittlung des Starts einer Bedingung (Steigende Flanke)

Um die Aufgabe "Ermittlung des Starts einer Bedingung (Steigende Flanke)" separate anzugehen, erzeugen wir uns übersichtliche Beispieldaten:
- ein Array mit 7 Elementen als Zeitachse und
- ein weiteres Array gleicher Länge als die Werte, wobei die Zeitpunkte 2,3 und 4 auf 1 stehen.

In [ ]:
t_test = np.arange(7, dtype=float)
x_test = 1.0*((t_test>1)&(t_test<5))
print(f"t_test = {t_test}")
print(f"x_test = {x_test}")

Wir wollen jetzt bestimmen, bei welchem Zeitpunkt, die Werte das erste Mal auf 1 gehen.
<br>
Erwartete Lösung: Zeitpunt 2
<br>
<br>
Dazu formulieren wir die Bedingung als eine Maske:

In [ ]:
mask = x_test>0.5
mask

Ausgabe der Zeitpunkte, wo die Bedingung erfüllt ist:

In [ ]:
t_test[mask]

Durch Auswahl des ersten Wertes bestimmen wir die steigende Flanke:

In [ ]:
t_test[mask][0]

Jetzt wollen dies auf das Signal `BrakeSwitch` anwenden und damit den Startzeitpunkt bestimmen, wenn der Fahrer auf das Bremspedal tritt:

In [ ]:
mask=BrakeSwitch>0.5
mask

In [ ]:
t0 = t[mask][0]
t0

In [ ]:
print(f"Start der Bremsung durch den Fahrer bei t0 = {t0:4.2f} s")

Welche Geschwindigkeit hat das Fahrzeug?

In [ ]:
v0 = v[mask][0]
v0

In [ ]:
print(f"Beim Start der Bremsung fährt das Fahrzeug v0 = {v0:4.2f} km/h")

Welche Weg wurde bis dahin zurückgelegt?

In [ ]:
s0 = s_vom_start_der_Messung[mask][0]
s0

In [ ]:
print(f"Beim Start der Bremsung hat das Fahrzeug s0 = {s0:4.2f} m")

### Bestimmung des Fahrzeugstillstands

Für eine vereinfachte Stillstandserkennung, betrachten wir den Zeitpunkt, wo die Geschwindigkeit unter einen sehr kleinen Wert fällt.

In [ ]:
mask = v<0.01
t_end = t[mask][0]
v_end = v[mask][0]
s_end = s_vom_start_der_Messung[mask][0]
t_end,v_end,s_end

In [ ]:
Bremsweg = s_end - s0
Bremsweg

### Weg relativ zum Start der Messung; davor Weg auf Null setzen

In [ ]:
s = s_vom_start_der_Messung - s0
mask = t<t0
s[mask] = 0

### Bestimmung von vb und sb

Zeitpunkt, wo die Fahrzeugeschwindigkeit auf 0.8 der Geschwindig zu Beginn der Bremsung reduziert worden ist.

In [ ]:
vb = 0.8*v0
print(f"vb: {vb} km/h")
mask = v<=vb
t_vb     = t[mask][0]
vb_value = v[mask][0]
s_b      = s[mask][0]
print(f"t_vb:{t_vb}s vb_value:{vb_value:4.2f} km/h s_b:{s_b:4.2f}m")

Wir machen eine Funktion daraus, weil wir es für ve und se wiederverwenden wollen:

In [ ]:
def Weg_zu_vorgegebener_Geschwindigkeit(t,v,v0,relative_Schwelle):
    v_Schwellewert = relative_Schwelle*v0
    Schwelle_unterschritten_mask = v<=v_Schwellewert
    t_Schwelle = t[Schwelle_unterschritten_mask][0]
    v_Schwelle = v[Schwelle_unterschritten_mask][0]
    s_Schwelle = s[Schwelle_unterschritten_mask][0]
    print(f'Abweichung der Schwelle {v_Schwellewert - v_Schwelle:4.2f} km/h')
    return t_Schwelle,v_Schwelle,s_Schwelle

In [ ]:
t_vb, vb, sb = Weg_zu_vorgegebener_Geschwindigkeit(t,v,v0,0.8)
t_vb, vb, sb

In [ ]:
t_ve, ve, se = Weg_zu_vorgegebener_Geschwindigkeit(t,v,v0,0.1)
t_ve, ve, se

### Berechnen der mittlere Vollverzögerung

In [ ]:
# mittlere Vollverzögerung
d_m = (vb**2-ve**2)/(25.92*(se-sb))
d_m

In [ ]:
print(f"Die mittlere Vollverzögerung beträgt: d_m = {d_m:4.2f} m/s^2")

### Ermittlung des Bremswegs

In [ ]:
t_stillstand, v_stillstand, s_stillstand = Weg_zu_vorgegebener_Geschwindigkeit(t,v,v0,0.01) # fast Stillstand
t_stillstand, v_stillstand, s_stillstand

In [ ]:
Bremsweg = s_stillstand
Bremsweg

### Plausibilisierung - Bremsweg mit mittlerer Verzögerung
$$
    s = \frac{1}{2} \cdot a \cdot t^2
$$
$$
     v = a \cdot t  \Rightarrow  t = \frac{v}{a}
$$

$$
    s = \frac{1}{2} \cdot a \cdot t^2 = \frac{1}{2} \cdot \frac{v^2}{a} = \frac{v^2}{2a}
$$

In [ ]:
(v0/3.6)**2/2/d_m

Der mit der mittleren Verzögerung ermittelte Bremsweg stimmt mit dem weiter oben ermittelten Bremsweg gut überein.

## Schritt 7: Plot für Bericht erstellen

In [ ]:
fig,(ax1,ax2,ax3) = plt.subplots(ncols=1,nrows=3, figsize=(12,8), sharex=True,constrained_layout=True)

fig.suptitle("ABS-Bremsung \n"
             f"Ausganggeschwindigkeit: {v0:3.1f} km/h"
             f" - Bremsweg: {Bremsweg:3.1f} m"
             f" - mittlere Vollverzögerung: {d_m:3.1f} m/s$^2$", 
             fontsize=16)

# ---------------------------------------
# erster Subplot - Bremsschalter und Bremspedal
ax1.set_title('Bremsschalter und Bremspedal')
ax1.plot(t-t0,BrakeSwitch*100,label='Bremsschalter *100')
ax1.plot(t-t0,BrkPedPos,label='Bremspedal 0..100%')
ax1.legend(loc='upper right')
ax1.grid(True)
ax1.set_ylabel('[%]')
ax1.set_ylim([-10,110])


# -------------------------------------
# zweiter Subplot
# Überschrift
ax2.set_title('Fahrzeuggeschwindigkeit')

# Signale
ax2.plot(t-t0,v,label='Fahrzeuggeschwindigkeit')

# Markierungen
ax2.plot(t0-t0,v0,marker='x',color='red')                      # v0
ax2.plot(t_vb-t0,vb,marker='x',color='red')                    # vb
ax2.plot(t_ve-t0,ve,marker='x',color='red')                    # ve
ax2.plot(t_stillstand-t0,v_stillstand,marker='x',color='red')  # Stillstand

# horizontale Linien
ax2.axhline(y=v0,color='black',alpha=0.3,label=f'$v_0$ = {v0:3.1f} km/h',ls='-')
ax2.axhline(y=vb,color='black',alpha=0.3,label=f'$v_b$ = 0.8 $v_0$ = {vb:3.1f} km/h',ls='--')
ax2.axhline(y=ve,color='black',alpha=0.3,label=f'$v_e$ = 0.1 $v_0$ = {ve:3.1f} km/h',ls='-.')

# vertikale Linien
ax2.axvline(x=t0-t0,color='black',alpha=0.3,ls='--')
ax2.axvline(x=t_vb-t0,color='black',alpha=0.3,ls='--')
ax2.axvline(x=t_ve-t0,color='black',alpha=0.3,ls='--')
ax2.axvline(x=t_stillstand-t0,color='black',alpha=0.3,ls='--')

ax2.legend(loc='upper right')
ax2.grid(True)
ax2.set_ylabel('[km/h]')
ax2.set_ylim([-10,90])

# ----------------------------------
# dritter Subplot
ax3.set_title('Bremsweg')
ax3.plot(t-t0,s,label='Bremsweg')

# Markierungen
ax3.plot(t0-t0,0,marker='x',color='red')
ax3.plot(t_vb-t0,sb,marker='x',color='red')
ax3.plot(t_ve-t0,se,marker='x',color='red')
ax3.plot(t_stillstand-t0,s_stillstand,marker='x',color='red')

# horizontale Linien
ax3.axhline(y=se,color='black',alpha=0.3,label=f'se = {se:3.1f} m',ls='-.')
ax3.axhline(y=sb,color='black',alpha=0.3,label=f'sb = {sb:3.1f} m',ls='--')


# vertikale Linien
ax3.axvline(x=t0-t0,color='black',alpha=0.3,ls='--')
ax3.axvline(x=t_vb-t0,color='black',alpha=0.3,ls='--')
ax3.axvline(x=t_ve-t0,color='black',alpha=0.3,ls='--')
ax3.axvline(x=t_stillstand-t0,color='black',alpha=0.3,ls='--')


ax3.legend(loc='upper right')
ax3.grid(True)
ax3.set_ylabel('[m]')
ax3.set_ylim([-5,50])

ax3.set_xlabel('Zeit [s]')

ax3.set_xlim([-1,t_end+1-t0])

# Abspeichern als  png-Bild
plt.savefig('ABS-Bremsung-Bremsweg Auswertung.png', facecolor='white')